In [3]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 83.5 MB/s eta 0:00:00


In [10]:
import gurobipy as gp
from gurobipy import GRB

In [9]:
# definiamo le variabili
J = ["A", "B", "C", "D", "E"]
# definiamo i costi
c = dict(zip(J, [400, 300, 500, 200, 600]))
# definiamo i benefici
b = dict(zip(J, [8, 6, 9, 5, 11]))
# budget massimo disponibile: vincolo
budget = 1000

# instanziamo il modello
m = gp.Model("attivita")
# aggiunta di variabili
x = m.addVars(J, vtype=GRB.BINARY, name="x")
# definizione della funzione obiettivo
m.setObjective(gp.quicksum(b[j] * x[j] for j in J), GRB.MAXIMIZE)
# aggiungiamo il vincolo di budget
m.addConstr(gp.quicksum(c[j] * x[j] for j in J) <= budget, name="budget")
# chiamiamo il risolutore
m.optimize()

# verifichiamo l'esito dell'esecuzione
print("Stato:", m.Status)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 1 rows, 5 columns and 5 nonzeros (Max)
Model fingerprint: 0xb4325357
Model has 5 linear objective coefficients
Variable types: 0 continuous, 5 integer (5 binary)
Coefficient statistics:
  Matrix range     [2e+02, 6e+02]
  Objective range  [5e+00, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+03, 1e+03]

Found heuristic solution: objective 19.0000000
Presolve removed 1 rows and 5 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 2: 20 19 

Optimal solution found (tolerance 1.00e-04)
Best objective 2.000000000000e+01, best bound 2.000000000000e+01, gap 0.0000%
Stat

In [8]:
# se una soluzione esiste, allora entriamo nella condizione if
if m.SolCount > 0:
  # recuperiamo la soluzione ottima
  # le variabili sono in {0,1}, quindi X>0.5 recupera esattamente le variabili settate ad 1
  scelti = [j for j in J if x[j].X > 0.5]
  # costo della soluzione ottima
  costo = sum(c[j] for j in scelti)
  # beneficio della soluzione ottima
  beneficio = sum(b[j] for j in scelti)
  # stampa a video
  print(scelti, "Costo:", costo, "Beneficio:", beneficio)

  # sanity check: il costo deve essere minore del vincolo di budget
  assert costo <= budget
  # altro sanity check: il valore obiettivo ricostruito deve concordare
  # con quello trovato dal risolutore, entro una certa tolleranza
  assert abs(beneficio - m.ObjVal) <= 1e-6
  if m.Status == GRB.OPTIMAL:
    print("Ottimo certificato entro le tolleranze.")
  else:
    print("Soluzione disponibile: controllare lo stato.")
else:
  print("Nessuna soluzione disponibile: controllare lo stato.")

# elimina il modello, liberando memoria e risorse
m.dispose()

['B', 'C', 'D'] Costo: 1000 Beneficio: 20
Ottimo certificato entro le tolleranze.


In [11]:
# last sanity check: binary variables
m.X

[0.0, 1.0, 1.0, 1.0, 0.0]